# Scholarship + College Research Agent

Build a hybrid-retrieval, tool-loadout RAG assistant for high-school juniors/seniors and gap-year applicants researching scholarships and colleges. It grounds answers in the two local metadata corpora, cites source URLs, treats missing data as unknown, and does not submit applications or retain student profiles.

## Goals
- Ingest two local metadata corpora into one Pinecone index with `scholarships` and `colleges` namespaces.
- Combine dense Nebius embeddings with local BM25 using reciprocal-rank fusion.
- Expose six typed LangChain tools and select a query-specific subset before agent invocation.
- Compare tool-loadout runs with all six tools and keep human review in the evaluation loop.

## Task checklist
| Task | Implemented in notebook cells |
|---|---|
| Task 1 - LangChain documents, structure-aware chunking, Nebius embeddings | 4 |
| Task 2 - Six typed, evidence-returning tools | 7 |
| Task 3 - Idempotent Pinecone ingestion, namespaces, local BM25, dry run | 5 |
| Task 4 - Required system prompt and rules | 8 |
| Task 5 - Nebius `create_agent`, recursion limit, graceful errors | 10 |
| Task 6 - Hybrid retrieval and tool loadout | 6, 9 |
| Task 7 - 18 demos including college research, evaluation harness, all-tools comparison | 11, 12 |

# 1. Architecture

![Architecture flowchart: offline ingestion and online question answering](scholarship_college_architecture.png)

The diagram is embedded as an image so it renders directly in notebook viewers that do not support Mermaid.

# 2. Setup and configuration

Run this notebook with the STEM project folder as the working directory. `DRY_RUN=True` is deliberately the default: corpus loading, normalization, chunking, and BM25 construction are local; no embedding, Pinecone, Tavily, or LLM request is made until the corresponding flag/agent call is explicitly enabled. The API model-list check and one embedding probe are performed only when `DRY_RUN=False`.

LangChain agent construction follows the `create_agent` style in the tool-loadout reference notebook. The requested reference `app.py` was not present; the available `Customer_Support_Agent_Tool_Loadout/app.ipynb` contains the requested “Building the Agent”, “Selecting Tools and Handling a Request”, and “Running Without Tool Loadout (All Tools)” patterns.

In [1]:
%pip install -U langchain-pinecone

import hashlib
import ipaddress
import json
import logging
import math
import os
import re
import time
from datetime import date, datetime, timedelta
from pathlib import Path
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser

import os
import json
from dotenv import load_dotenv
from langchain_community.retrievers import BM25Retriever
from langchain_pinecone import PineconeVectorStore
import pandas as pd
import requests
from bs4 import BeautifulSoup
from langchain.agents import create_agent
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import Pinecone, ServerlessSpec


logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logger = logging.getLogger("scholarship_college_rag")
load_dotenv()
BASE_DIR = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "stem_corpus_output" / "data" / "scholarship_metadata.json").is_file()
    ),
    Path.cwd(),
)
SCHOLARSHIP_PATH = BASE_DIR / "stem_corpus_output" / "data" / "scholarship_metadata.json"
COLLEGE_PATH = BASE_DIR / "stem_corpus_output" / "data" / "college_metadata.json"
EVALUATION_PATH = BASE_DIR / "reports" / "rag_evaluation.csv"

ENV_PATH = BASE_DIR / ".env"
load_dotenv(ENV_PATH, override=True)
DRY_RUN = False
RUN_DEMOS = True
USE_TAVILY = False
CONFIRM_DELETE_NAMESPACES = False
PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "stem-scholarship-college-rag").strip()
PINECONE_CLOUD = os.getenv("PINECONE_CLOUD", "aws")
PINECONE_REGION = os.getenv("PINECONE_REGION", "us-east-1")
NEBIUS_BASE_URL = os.getenv("NEBIUS_BASE_URL", "").strip().rstrip("/")
NEBIUS_API_KEY = os.getenv("NEBIUS_API_KEY", "").strip()
NEBIUS_LLM_MODEL = os.getenv("NEBIUS_LLM_MODEL", "").strip()
NEBIUS_EMBED_MODEL = os.getenv("NEBIUS_EMBED_MODEL", "").strip()
print(f"Loaded project .env: {ENV_PATH.is_file()}; embedding model configured: {bool(NEBIUS_EMBED_MODEL)}")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY", "").strip()

assert SCHOLARSHIP_PATH.is_file(), f"Missing corpus: {SCHOLARSHIP_PATH}"
assert COLLEGE_PATH.is_file(), f"Missing corpus: {COLLEGE_PATH}"
print(f"Scholarship corpus: {SCHOLARSHIP_PATH}")
print(f"College corpus: {COLLEGE_PATH}")
print(f"DRY_RUN={DRY_RUN}; RUN_DEMOS={RUN_DEMOS}; USE_TAVILY={USE_TAVILY}")

Note: you may need to restart the kernel to use updated packages.


C:\Users\Rosh\AppData\Local\Temp\ipykernel_3548\39410389.py:19: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever
c:\Users\Rosh\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\Rosh\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Loaded project .env: True; embedding model configured: True
Scholarship corpus: c:\Users\Rosh\GENAI\WEEK2_PROJECT\college_resources_for_high_schoolers_STEM\stem_corpus_output\data\scholarship_metadata.json
College corpus: c:\Users\Rosh\GENAI\WEEK2_PROJECT\college_resources_for_high_schoolers_STEM\stem_corpus_output\data\college_metadata.json
DRY_RUN=False; RUN_DEMOS=True; USE_TAVILY=False


# 3. Data loading and schema inspection

Verified local snapshot (recomputed by the next cell when run):

- Scholarship file: **76 records**, 30 actual fields; 69 records have `page_type="overview"`, 7 have `page_type="not_a_program"`, and 72 have a non-null `program_name`. This includes duplicate program names/source pages; do not assume one record per award.
- Scholarship fields: `accepts_target_applicants`, `applicant_stage`, `award_amount_max_usd`, `award_amount_min_usd`, `award_amount_text`, `award_type`, `citizenship_requirements`, `crawl_date`, `deadline_date_iso`, `deadline_recurs_annually`, `deadline_text`, `demographic_focus`, `eligibility_summary`, `essay_required`, `evidence`, `gpa_minimum`, `merit_based`, `need_based`, `organization`, `page_type`, `program_name`, `recommendation_letters`, `renewable`, `residency_restriction`, `source_url`, `stem_fields`, `stem_relevance`, `stem_relevance_evidence`, `target_applicant_evidence`, `tier`.
- Key scholarship null/blank rates: minimum award 30/76 (39.5%); maximum 25/76 (32.9%); award text 16/76 (21.1%); renewable 58/76 (76.3%); deadline text 41/76 (53.9%); parsed deadline 52/76 (68.4%); recurrence flag 65/76 (85.5%); GPA minimum 60/76 (78.9%); citizenship 62/76 (81.6%); residency 51/76 (67.1%). Parsed deadline range: **2026-02-15 through 2027-06-30**. There is no dedicated `renewal_terms` field in the actual schema.
- College file is a wrapper (`schema_version`, `generated_at`, `source_file`, `source_sha256`, `record_count`, `extraction_method`, `notes`, `colleges`) with **2,191** records in `colleges`. Record fields: `admission_rate_percent`, `admissions_data_year`, `admissions_selectivity_band`, `admissions_selectivity_status`, `admissions_source_url`, `admissions_url`, `admitted_imputation_flag`, `applicants_imputation_flag`, `application_url`, `city`, `college_name`, `control`, `crawl_focus`, `crawl_seed_url`, `directory_data_year`, `directory_source_url`, `extracted_date`, `financial_aid_url`, `funding_verification`, `net_price_calculator_url`, `opportunity_verification_status`, `scholarships_url`, `source_agency`, `source_category`, `source_type`, `state`, `stem_program_verification`, `target_applicant_stage`, `tier`, `tier_basis`, `tier_rules`, `tier_status`, `undergraduate_admitted`, `undergraduate_applicants`, `unitid`, `url_verification_status`, `website_domain`, `website_url`, `zip_code`.
- Key college null/blank rates: website URL 1/2,191 (0.05%); financial-aid URL 133/2,191 (6.1%); admissions URL 119/2,191 (5.4%); scholarships URL 2,191/2,191 (100%); admission-rate percent 587/2,191 (26.8%). STEM-program fields are populated with **pending-verification** text, not verified program inventories. College inclusion is a discovery seed, not evidence of a STEM program, award, admission chance, or scholarship availability.
- `college_metadata.json` has no deadline fields. Null rates above count null/blank scalars; empty lists are inspected separately by the code.

In [2]:
with SCHOLARSHIP_PATH.open(encoding="utf-8") as f:
    scholarship_records = json.load(f)
with COLLEGE_PATH.open(encoding="utf-8") as f:
    college_payload = json.load(f)
college_records = college_payload["colleges"]

def schema_report(records, label, key_fields):
    fields = sorted({key for row in records for key in row})
    print(f"\n{label}: {len(records):,} records")
    print("Fields:", ", ".join(fields))
    for key in key_fields:
        if key in fields:
            missing = sum(row.get(key) is None or row.get(key) == "" for row in records)
            empty_lists = sum(row.get(key) == [] for row in records)
            print(f"{key}: null/blank={missing}/{len(records)}; empty-list={empty_lists}/{len(records)}")
    return fields

scholarship_fields = schema_report(
    scholarship_records,
    "Scholarships",
    ["award_amount_min_usd", "award_amount_max_usd", "award_amount_text",
     "renewable", "deadline_text", "deadline_date_iso",
     "deadline_recurs_annually", "gpa_minimum", "citizenship_requirements",
     "residency_restriction", "program_name"],
)
college_fields = schema_report(
    college_records,
    "Colleges",
    ["college_name", "state", "website_url", "financial_aid_url",
     "admissions_url", "scholarships_url", "admission_rate_percent",
     "stem_program_verification", "funding_verification"],
)
deadline_dates = [
    date.fromisoformat(row["deadline_date_iso"])
    for row in scholarship_records
    if row.get("deadline_date_iso")
]
print("Page types:", pd.Series([r.get("page_type") for r in scholarship_records]).value_counts(dropna=False).to_dict())
print("Parsed deadline range:", min(deadline_dates), "to", max(deadline_dates))
print("College wrapper record_count:", college_payload.get("record_count"))


Scholarships: 76 records
Fields: accepts_target_applicants, applicant_stage, award_amount_max_usd, award_amount_min_usd, award_amount_text, award_type, citizenship_requirements, crawl_date, deadline_date_iso, deadline_recurs_annually, deadline_text, demographic_focus, eligibility_summary, essay_required, evidence, gpa_minimum, merit_based, need_based, organization, page_type, program_name, recommendation_letters, renewable, residency_restriction, source_url, stem_fields, stem_relevance, stem_relevance_evidence, target_applicant_evidence, tier
award_amount_min_usd: null/blank=30/76; empty-list=0/76
award_amount_max_usd: null/blank=25/76; empty-list=0/76
award_amount_text: null/blank=16/76; empty-list=0/76
renewable: null/blank=58/76; empty-list=0/76
deadline_text: null/blank=41/76; empty-list=0/76
deadline_date_iso: null/blank=52/76; empty-list=0/76
deadline_recurs_annually: null/blank=65/76; empty-list=0/76
gpa_minimum: null/blank=60/76; empty-list=0/76
citizenship_requirements: null/

# 4. Structure-aware chunking and embeddings

Each scholarship or college record becomes a readable LangChain `Document`; college rows use the real `college_name` and directory/admissions/aid fields. Long records alone are split, with a repeated `[Program] [Organization] | [Section]` context header. The word count is a lightweight token estimate for reporting; the splitter uses a conservative character target (about 400 words/tokens, depending on content). Chunk IDs hash `source_url + section + chunk index`; including the record name in `section` keeps same-URL, different-program rows distinct and re-ingestion stable.

At configuration time, the notebook checks Nebius `/models` for the configured chat and embedding model IDs and obtains the embedding dimension from a live probe vector. This is only invoked when `DRY_RUN=False`; the dimension is never hardcoded.

In [3]:
TARGET_WORDS = 400
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1600,
    chunk_overlap=180,
    add_start_index=True,
    separators=["\n\n", "\n", ". ", " ", ""],
)

def safe_metadata_value(value):
    if value is None:
        return None
    if isinstance(value, (str, bool, int, float)):
        return value
    if isinstance(value, list):
        return [str(item) for item in value if item is not None]
    return json.dumps(value, ensure_ascii=False, sort_keys=True)

def readable_record(record, corpus):
    if corpus == "scholarships":
        keys = [
            "program_name", "organization", "page_type", "award_type",
            "award_amount_min_usd", "award_amount_max_usd", "award_amount_text",
            "renewable", "deadline_text", "deadline_date_iso",
            "deadline_recurs_annually", "eligibility_summary", "applicant_stage",
            "stem_fields", "gpa_minimum", "citizenship_requirements",
            "residency_restriction", "demographic_focus", "need_based",
            "merit_based", "essay_required", "crawl_date", "evidence",
        ]
        name = str(record.get("program_name") or "Unnamed scholarship record")
        organization = str(record.get("organization") or "Organization unstated")
        section = f"{record.get('page_type') or 'program'} | {name}"
        source_url = record.get("source_url") or ""
    else:
        keys = [
            "college_name", "city", "state", "control", "website_url",
            "financial_aid_url", "admissions_url", "application_url",
            "net_price_calculator_url", "scholarships_url",
            "admission_rate_percent", "admissions_data_year",
            "stem_program_verification", "funding_verification",
            "opportunity_verification_status", "directory_source_url",
            "admissions_source_url", "target_applicant_stage",
        ]
        name = str(record.get("college_name") or "Unnamed college record")
        organization = name
        section = f"college_directory | {name}"
        source_url = (
            record.get("website_url")
            or record.get("directory_source_url")
            or record.get("admissions_source_url")
            or ""
        )
    lines = []
    for key in keys:
        value = record.get(key)
        if value is not None and value != "" and value != []:
            rendered = json.dumps(value, ensure_ascii=False) if isinstance(value, (list, dict)) else str(value)
            lines.append(f"{key}: {rendered}")
    metadata = {
        key: safe_metadata_value(value)
        for key, value in record.items()
        if safe_metadata_value(value) is not None
    }
    metadata.update({
        "corpus": corpus,
        "display_name": name,
        "organization": organization,
        "section": section,
        "source_url": str(source_url),
    })
    metadata["metadata_json"] = json.dumps(metadata, ensure_ascii=False, sort_keys=True)
    return name, organization, section, source_url, "\n".join(lines), metadata

def chunk_records(records, corpus):
    documents = []
    for record in records:
        name, organization, section, source_url, body, metadata = readable_record(record, corpus)
        header = f"[{name}] [{organization}] | {section}"
        source_doc = Document(page_content=body, metadata=metadata)
        parts = splitter.split_documents([source_doc]) if len(body.split()) > TARGET_WORDS else [source_doc]
        for index, part in enumerate(parts):
            chunk_id = hashlib.sha256(
                f"{source_url}|{section}|{index}".encode("utf-8")
            ).hexdigest()
            chunk_metadata = dict(part.metadata)
            chunk_metadata["chunk_id"] = chunk_id
            chunk_metadata["chunk_index"] = index
            chunk_metadata["estimated_tokens"] = max(1, math.ceil(len(part.page_content.split()) * 1.3))
            documents.append(Document(
                page_content=f"{header}\n{part.page_content}",
                metadata=chunk_metadata,
            ))
    return documents

chunks_by_namespace = {
    "scholarships": chunk_records(scholarship_records, "scholarships"),
    "colleges": chunk_records(college_records, "colleges"),
}

def chunk_stats(documents):
    sizes = sorted(doc.metadata["estimated_tokens"] for doc in documents)
    if not sizes:
        return {"chunks": 0, "min_estimated_tokens": 0, "median_estimated_tokens": 0, "max_estimated_tokens": 0}
    middle = len(sizes) // 2
    median = sizes[middle] if len(sizes) % 2 else (sizes[middle - 1] + sizes[middle]) / 2
    return {"chunks": len(sizes), "min_estimated_tokens": sizes[0], "median_estimated_tokens": median, "max_estimated_tokens": sizes[-1]}

print({namespace: chunk_stats(docs) for namespace, docs in chunks_by_namespace.items()})

{'scholarships': {'chunks': 76, 'min_estimated_tokens': 17, 'median_estimated_tokens': 126.0, 'max_estimated_tokens': 182}, 'colleges': {'chunks': 2191, 'min_estimated_tokens': 76, 'median_estimated_tokens': 88, 'max_estimated_tokens': 99}}


In [4]:
EMBEDDING_DIM = None
EMBEDDINGS = None
PINECONE_CLIENT = None
PINECONE_INDEX = None

def required_env(name):
    value = os.getenv(name, "").strip()
    if not value:
        raise RuntimeError(f"Set {name} in the local .env file before enabling API operations.")
    return value

def get_embeddings():
    global EMBEDDINGS
    if EMBEDDINGS is None:
        EMBEDDINGS = OpenAIEmbeddings(
            model=NEBIUS_EMBED_MODEL,
            dimensions=512,
            api_key=required_env("NEBIUS_API_KEY").strip(),
            base_url="https://api.tokenfactory.nebius.com/v1/",
            check_embedding_ctx_length=False,)
           
    return EMBEDDINGS

def verify_nebius_models_and_dimension():
    base_url = required_env("NEBIUS_BASE_URL").rstrip("/")
    response = requests.get(
        f"{base_url}/models",
        headers={"Authorization": f"Bearer {required_env('NEBIUS_API_KEY')}"},
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()
    model_rows = payload if isinstance(payload, list) else payload.get("data", [])
    model_ids = {row.get("id") for row in model_rows if isinstance(row, dict)}
    chat_name = required_env("NEBIUS_LLM_MODEL")
    embed_name = required_env("NEBIUS_EMBED_MODEL")
    missing = {chat_name, embed_name} - model_ids
    if missing:
        raise ValueError(f"Configured model(s) not listed by Nebius /models: {sorted(missing)}")
    dimension = len(get_embeddings().embed_query("embedding dimension probe"))
    if dimension <= 0:
        raise ValueError("Nebius embedding probe returned an empty vector.")
    print(f"Verified model IDs: {chat_name}, {embed_name}; probe dimension: {dimension}")
    return dimension

def get_embedding_dimension():
    global EMBEDDING_DIM
    if EMBEDDING_DIM is None:
        EMBEDDING_DIM = verify_nebius_models_and_dimension()
    return EMBEDDING_DIM

if DRY_RUN:
    print("Dry run: skipped Nebius model listing and embedding probe; dimension remains unknown.")
else:
    get_embedding_dimension()

INFO HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"


Verified model IDs: openai/gpt-oss-120b, Qwen/Qwen3-Embedding-8B; probe dimension: 512


# 5. Pinecone ingestion

`DRY_RUN=True` prints chunk counts and estimated embedding-token volume, without creating an index or sending vectors. Set it to `False` only after checking `.env`, model IDs, region, credits, and the intended index name. Ingestion skips existing chunks only when their stored text matches; changed text is re-embedded and upserted under the same ID.

In [5]:
print(PINECONE_INDEX)

None


In [6]:
UPSERT_BATCH_SIZE = 100
NAMESPACES = ("scholarships", "colleges")

def connect_pinecone():
    global PINECONE_CLIENT, PINECONE_INDEX
    if PINECONE_INDEX is not None:
        return PINECONE_INDEX
    PINECONE_CLIENT = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
    names = set(PINECONE_CLIENT.list_indexes().names())
    dimension = get_embedding_dimension()
    if PINECONE_INDEX_NAME not in names:
        PINECONE_CLIENT.create_index(
            name=PINECONE_INDEX_NAME,
            dimension=dimension,
            metric="cosine"
            #,spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
        )
        while not PINECONE_CLIENT.describe_index(PINECONE_INDEX_NAME).status["ready"]:
            time.sleep(2)
    description = PINECONE_CLIENT.describe_index(PINECONE_INDEX_NAME)
    if description.dimension != dimension or description.metric != "cosine":
        raise ValueError(
            f"Index dimension/metric mismatch: index={description.dimension}/{description.metric}, "
            f"embedding={dimension}/cosine"
        )
    PINECONE_INDEX = PINECONE_CLIENT.Index(host=description.host)
    return PINECONE_INDEX

def pinecone_metadata(metadata):
    safe = {}
    for key, value in metadata.items():
        value = safe_metadata_value(value)
        if value is not None:
            safe[key] = value
    return safe

def ingest_namespace(namespace, documents):
    index = connect_pinecone()
    upserted = 0
    updated_existing = 0
    skipped_unchanged = 0
    for start in range(0, len(documents), UPSERT_BATCH_SIZE):
        batch = documents[start:start + UPSERT_BATCH_SIZE]
        ids = [doc.metadata["chunk_id"] for doc in batch]
        existing = index.fetch(ids=ids, namespace=namespace).vectors
        changed_docs = []
        for doc in batch:
            old = existing.get(doc.metadata["chunk_id"])
            if old is None:
                changed_docs.append(doc)
                continue
            old_metadata = old.get("metadata", {}) if isinstance(old, dict) else getattr(old, "metadata", {})
            if (old_metadata or {}).get("page_content") != doc.page_content:
                changed_docs.append(doc)
                updated_existing += 1
            else:
                skipped_unchanged += 1
        if not changed_docs:
            continue
        vectors = get_embeddings().embed_documents([doc.page_content for doc in changed_docs])
        rows = [
            {
                "id": doc.metadata["chunk_id"],
                "values": vector,
                "metadata": {
                    **pinecone_metadata(doc.metadata),
                    "page_content": doc.page_content,
                },
            }
            for doc, vector in zip(changed_docs, vectors, strict=True)
        ]
        index.upsert(vectors=rows, namespace=namespace)
        upserted += len(rows)
    return {
        "namespace": namespace,
        "upserted": upserted,
        "updated_existing": updated_existing,
        "skipped_unchanged": skipped_unchanged,
    }

if DRY_RUN:
    summary = {}
    for namespace, documents in chunks_by_namespace.items():
        tokens = sum(doc.metadata["estimated_tokens"] for doc in documents)
        summary[namespace] = {
            "chunks": len(documents),
            "estimated_embedding_tokens": tokens,
            "would_upsert_at_most": len(documents),
        }
    print("DRY RUN - no API requests made:", json.dumps(summary, indent=2))
else:
    ingestion_summary = [
        ingest_namespace(namespace, chunks_by_namespace[namespace])
        for namespace in NAMESPACES
    ]
    print(json.dumps(ingestion_summary, indent=2))


INFO HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
INFO HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"


[
  {
    "namespace": "scholarships",
    "upserted": 1,
    "updated_existing": 1,
    "skipped_unchanged": 75
  },
  {
    "namespace": "colleges",
    "upserted": 1,
    "updated_existing": 1,
    "skipped_unchanged": 2190
  }
]


In [7]:
# Destructive operation: keep both guards false unless namespace deletion is intentional.
if CONFIRM_DELETE_NAMESPACES:
    if DRY_RUN:
        raise RuntimeError("Set DRY_RUN=False as a second explicit confirmation before deletion.")
    index = connect_pinecone()
    for namespace in NAMESPACES:
        index.delete(delete_all=True, namespace=namespace)
    print(f"Deleted vectors in namespaces: {NAMESPACES}")
else:
    print("Namespace deletion skipped; set CONFIRM_DELETE_NAMESPACES=True and DRY_RUN=False to enable.")

Namespace deletion skipped; set CONFIRM_DELETE_NAMESPACES=True and DRY_RUN=False to enable.


# 6. Hybrid retriever

Dense Pinecone and local BM25 candidates are combined with reciprocal-rank fusion (`1 / (60 + rank)`). The result retains the fused score plus dense/BM25 scores and metadata. `TOP_K=5` is the final result count; state filtering on scholarships is evaluated locally against actual `residency_restriction` text because the scholarship schema has no `state` column.

In [8]:
TOP_K = 5
RRF_CONSTANT = 60
STATE_NAMES = {
    "AL": "Alabama", "AK": "Alaska", "AZ": "Arizona", "AR": "Arkansas",
    "CA": "California", "CO": "Colorado", "CT": "Connecticut", "DE": "Delaware",
    "DC": "District of Columbia", "FL": "Florida", "GA": "Georgia", "HI": "Hawaii",
    "ID": "Idaho", "IL": "Illinois", "IN": "Indiana", "IA": "Iowa",
    "KS": "Kansas", "KY": "Kentucky", "LA": "Louisiana", "ME": "Maine",
    "MD": "Maryland", "MA": "Massachusetts", "MI": "Michigan", "MN": "Minnesota",
    "MS": "Mississippi", "MO": "Missouri", "MT": "Montana", "NE": "Nebraska",
    "NV": "Nevada", "NH": "New Hampshire", "NJ": "New Jersey", "NM": "New Mexico",
    "NY": "New York", "NC": "North Carolina", "ND": "North Dakota", "OH": "Ohio",
    "OK": "Oklahoma", "OR": "Oregon", "PA": "Pennsylvania", "RI": "Rhode Island",
    "SC": "South Carolina", "SD": "South Dakota", "TN": "Tennessee", "TX": "Texas",
    "UT": "Utah", "VT": "Vermont", "VA": "Virginia", "WA": "Washington",
    "WV": "West Virginia", "WI": "Wisconsin", "WY": "Wyoming",
}
STATE_CODES = {name.casefold(): code for code, name in STATE_NAMES.items()}

def normalized_state(value):
    text = str(value).strip()
    if len(text) == 2 and text.upper() in STATE_NAMES:
        return text.upper()
    return STATE_CODES.get(text.casefold(), text)

bm25_by_namespace = {}
bm25_documents = {}

def tokens(text):
    return re.findall(r"[\w'-]+", text.casefold())

for namespace, documents in chunks_by_namespace.items():
    bm25_documents[namespace] = documents
    bm25_by_namespace[namespace] = BM25Retriever.from_documents(
        documents,
        preprocess_func=tokens,
    ).vectorizer

def metadata_matches(metadata, filters, namespace):
    for key, value in (filters or {}).items():
        if value is None:
            continue
        actual = metadata.get(key)
        if key == "state" and namespace == "scholarships":
            restriction = str(metadata.get("residency_restriction", "")).casefold()
            requested = str(value).casefold()
            full_name = STATE_NAMES.get(str(value).upper(), str(value)).casefold()
            if restriction and full_name not in restriction and requested not in restriction:
                return False
            if not restriction:
                return False
        elif isinstance(actual, list):
            wanted = value if isinstance(value, list) else [value]
            if not any(
                str(item).casefold() in str(actual_value).casefold()
                or str(actual_value).casefold() in str(item).casefold()
                for item in wanted
                for actual_value in actual
            ):
                return False
        elif actual is None:
            return False
        elif isinstance(value, bool):
            if actual is not value:
                return False
        elif str(value).casefold() not in str(actual).casefold():
            return False
    return True

def native_pinecone_filter(filters, namespace):
    clauses = {}
    for key, value in (filters or {}).items():
        if value is None or (key == "state" and namespace == "scholarships"):
            continue
        if key not in {"state", "stem_fields", "applicant_stage", "demographic_focus", "need_based", "essay_required"}:
            continue
        if key == "state":
            value = normalized_state(value)
        clauses[key] = {"$in": [value]} if key in {
            "stem_fields", "applicant_stage", "demographic_focus"
        } else {"$eq": value}
    return clauses or None

def retrieve(query, namespace, filters=None, k=TOP_K):
    if namespace not in NAMESPACES:
        raise ValueError(f"Unknown namespace: {namespace}")
    if k < 1:
        raise ValueError("k must be positive")
    candidates = {}
    if not DRY_RUN:
        index = connect_pinecone()
        response = index.query(
            namespace=namespace,
            vector=get_embeddings().embed_query(query),
            top_k=max(k * 5, k),
            include_metadata=True,
            filter=native_pinecone_filter(filters, namespace),
        )
        for rank, match in enumerate(response.matches, start=1):
            metadata = dict(match.metadata or {})
            if not metadata_matches(metadata, filters, namespace):
                continue
            candidates[match.id] = {
                "metadata": metadata,
                "page_content": metadata.get("page_content", ""),
                "dense_score": float(match.score or 0.0),
                "bm25_score": None,
                "rrf_score": 1 / (RRF_CONSTANT + rank),
            }

    docs = bm25_documents[namespace]
    bm25_scores = bm25_by_namespace[namespace].get_scores(tokens(query))
    eligible = [
        (index, float(score), doc)
        for index, (score, doc) in enumerate(zip(bm25_scores, docs, strict=True))
        if score > 0 and metadata_matches(doc.metadata, filters, namespace)
    ]
    eligible.sort(key=lambda item: item[1], reverse=True)
    for rank, (_, score, doc) in enumerate(eligible[:max(k * 5, k)], start=1):
        chunk_id = doc.metadata["chunk_id"]
        entry = candidates.setdefault(chunk_id, {
            "metadata": doc.metadata,
            "page_content": doc.page_content,
            "dense_score": None,
            "bm25_score": None,
            "rrf_score": 0.0,
        })
        entry["bm25_score"] = score
        entry["rrf_score"] += 1 / (RRF_CONSTANT + rank)
    results = sorted(candidates.items(), key=lambda item: item[1]["rrf_score"], reverse=True)[:k]
    return [
        {
            "chunk_id": chunk_id,
            "score": item["rrf_score"],
            "dense_score": item["dense_score"],
            "bm25_score": item["bm25_score"],
            "page_content": item["page_content"],
            "metadata": item["metadata"],
        }
        for chunk_id, item in results
    ]

print("BM25 is ready for both namespaces; dense search is enabled after Pinecone ingestion.")

BM25 is ready for both namespaces; dense search is enabled after Pinecone ingestion.


# 7. Tools

All tool results are compact JSON strings with corpus values, evidence where available, and `source_url`. Records with `page_type="not_a_program"` are never promoted as awards. The college corpus contains no verified STEM-major or scholarship inventory, so those questions must be qualified or abstained from.

In [9]:
def compact(value):
    return json.dumps(value, ensure_ascii=False, default=str)

def record_evidence(record):
    return {
        "eligibility_summary": record.get("eligibility_summary"),
        "evidence": record.get("evidence"),
        "target_applicant_evidence": record.get("target_applicant_evidence"),
        "stem_relevance_evidence": record.get("stem_relevance_evidence"),
    }

def matched_scholarship_rows(query, limit=5):
    query_terms = set(tokens(query))
    rows = []
    for record in scholarship_records:
        if record.get("page_type") == "not_a_program" or not record.get("program_name"):
            continue
        name = str(record.get("program_name", ""))
        overlap = len(query_terms & set(tokens(name)))
        if query.casefold() in name.casefold() or overlap:
            rows.append((overlap, record))
    rows.sort(key=lambda item: item[0], reverse=True)
    return [row for _, row in rows[:limit]]

def matched_college_rows(query, limit=5):
    query_terms = set(tokens(query))
    rows = []
    for record in college_records:
        name = str(record.get("college_name", ""))
        overlap = len(query_terms & set(tokens(name)))
        if query.casefold() in name.casefold() or overlap:
            rows.append((overlap, record))
    rows.sort(key=lambda item: item[0], reverse=True)
    return [row for _, row in rows[:limit]]

def retrieved_records(query, namespace, filters=None, limit=5):
    results = retrieve(query, namespace, filters=filters, k=limit)
    return [
        {
            "chunk_id": row["chunk_id"],
            "score": row["score"],
            "metadata": row["metadata"],
            "text": row["page_content"][:1800],
        }
        for row in results
    ]

In [10]:
@tool
def find_colleges(query: str, state: str | None = None, stem_field: str | None = None, max_results: int = 5) -> str:
    "Find college directory records by name or query; optional state and STEM-field filters."
    filters = {"state": state, "stem_fields": stem_field}
    hits = retrieve(query, "colleges", filters=filters, k=max_results)
    output = []
    for hit in hits:
        metadata = hit["metadata"]
        output.append({
            "college_name": metadata.get("college_name"),
            "city": metadata.get("city"),
            "state": metadata.get("state"),
            "website_url": metadata.get("website_url"),
            "admissions_url": metadata.get("admissions_url"),
            "financial_aid_url": metadata.get("financial_aid_url"),
            "scholarships_url": metadata.get("scholarships_url"),
            "stem_program_verification": metadata.get("stem_program_verification"),
            "funding_verification": metadata.get("funding_verification"),
            "source_url": metadata.get("website_url") or metadata.get("directory_source_url"),
            "evidence": {
                "directory_source_url": metadata.get("directory_source_url"),
                "admissions_source_url": metadata.get("admissions_source_url"),
                "opportunity_verification_status": metadata.get("opportunity_verification_status"),
            },
            "retrieval_score": hit["score"],
        })
    return compact({"results": output, "note": "Directory inclusion does not verify STEM programs, awards, admissions chances, or scholarship availability."})

@tool
def search_scholarships(query: str, stem_field: str | None = None, state: str | None = None, applicant_stage: str | None = None, demographic_focus: str | None = None, need_based: bool | None = None, max_results: int = 5) -> str:
    "Search scholarship program records with hybrid retrieval and supported metadata filters."
    filters = {
        "stem_fields": stem_field,
        "state": state,
        "applicant_stage": applicant_stage,
        "demographic_focus": demographic_focus,
        "need_based": need_based,
    }
    if "no essay" in query.casefold() or "without an essay" in query.casefold():
        filters["essay_required"] = False
    hits = retrieve(query, "scholarships", filters=filters, k=max_results)
    results = []
    for hit in hits:
        row = hit["metadata"]
        if row.get("page_type") == "not_a_program" or not row.get("program_name"):
            continue
        results.append({
            "program_name": row.get("program_name"),
            "organization": row.get("organization"),
            "award_type": row.get("award_type"),
            "award_amount_min_usd": row.get("award_amount_min_usd"),
            "award_amount_max_usd": row.get("award_amount_max_usd"),
            "award_amount_text": row.get("award_amount_text"),
            "deadline_text": row.get("deadline_text"),
            "deadline_date_iso": row.get("deadline_date_iso"),
            "renewable": row.get("renewable"),
            "applicant_stage": row.get("applicant_stage"),
            "stem_fields": row.get("stem_fields"),
            "essay_required": row.get("essay_required"),
            "crawl_date": row.get("crawl_date"),
            "source_url": row.get("source_url"),
            "evidence": record_evidence(row),
            "retrieval_score": hit["score"],
        })
    return compact({"results": results, "filters": filters, "note": "Missing or empty fields are unknown; records are not deduplicated across source pages."})

@tool
def evaluate_amount(program_name: str) -> str:
    "Return stored award ranges/text and renewal data for a named scholarship."
    rows = matched_scholarship_rows(program_name, limit=8)
    results = [{
        "program_name": row.get("program_name"),
        "organization": row.get("organization"),
        "award_amount_min_usd": row.get("award_amount_min_usd"),
        "award_amount_max_usd": row.get("award_amount_max_usd"),
        "award_amount_text": row.get("award_amount_text"),
        "renewable": row.get("renewable"),
        "renewal_terms": row.get("renewal_terms") or (row.get("evidence") or {}).get("renewal"),
        "eligibility_summary": row.get("eligibility_summary"),
        "amount_varies": (
            None if row.get("award_amount_min_usd") is None and row.get("award_amount_max_usd") is None
            else None if row.get("award_amount_min_usd") is None or row.get("award_amount_max_usd") is None
            else row.get("award_amount_min_usd") != row.get("award_amount_max_usd")
        ),
        "source_url": row.get("source_url"),
        "evidence": record_evidence(row),
        "crawl_date": row.get("crawl_date"),
    } for row in rows]
    return compact({"results": results, "note": "Null means the corpus does not establish an amount or renewal term."})

In [11]:
@tool
def evaluate_eligibility(program_name: str, student_profile: dict) -> str:
    "Compare an in-memory student profile with stored criteria; return met/not_met/unknown plus evidence."
    rows = matched_scholarship_rows(program_name, limit=5)
    if not rows:
        return compact({"program_name": program_name, "criteria": [], "note": "No matching program record found."})
    row = rows[0]
    evidence = record_evidence(row)
    criteria = []

    def add_criterion(name, value, outcome, evidence_value):
        criteria.append({"criterion": name, "student_value": value, "result": outcome, "evidence": evidence_value})

    grade = student_profile.get("grade")
    stages = row.get("applicant_stage") or []
    if grade is None or not stages:
        add_criterion("applicant_stage", grade, "unknown", stages or evidence.get("eligibility_summary"))
    else:
        grade_text = str(grade).casefold().replace("grade ", "")
        aliases = {"11": "junior", "12": "senior", "gap year": "gap_year", "gap-year": "gap_year"}
        grade_text = aliases.get(grade_text, grade_text)
        normalized_stages = {str(stage).casefold().replace(" ", "_") for stage in stages}
        add_criterion(
            "applicant_stage", grade,
            "met" if grade_text.replace(" ", "_") in normalized_stages else "not_met",
            stages,
        )

    gpa = student_profile.get("gpa")
    minimum = row.get("gpa_minimum")
    if gpa is None or minimum is None:
        add_criterion("gpa", gpa, "unknown", minimum)
    else:
        try:
            gpa_value = float(gpa)
            minimum_value = float(minimum)
            outcome = "unknown" if (minimum_value > 4 and gpa_value <= 4) else (
                "met" if gpa_value >= minimum_value else "not_met"
            )
            add_criterion("gpa", gpa, outcome, minimum)
        except (TypeError, ValueError):
            add_criterion("gpa", gpa, "unknown", minimum)

    major = student_profile.get("major")
    fields = row.get("stem_fields") or []
    if major is None or not fields:
        add_criterion("major", major, "unknown", fields or evidence.get("stem_relevance_evidence"))
    else:
        match = any(str(field).casefold() in str(major).casefold() or str(major).casefold() in str(field).casefold() for field in fields)
        add_criterion("major", major, "met" if match else "unknown", fields)

    state = student_profile.get("state")
    residence = row.get("residency_restriction")
    if state is None or not residence:
        add_criterion("residency", state, "unknown", residence)
    else:
        state_name = STATE_NAMES.get(str(state).upper(), str(state))
        add_criterion(
            "residency", state,
            "met" if state_name.casefold() in str(residence).casefold() else "unknown",
            residence,
        )

    citizenship = student_profile.get("citizenship")
    citizenship_rule = row.get("citizenship_requirements")
    if citizenship is None or not citizenship_rule:
        add_criterion("citizenship", citizenship, "unknown", citizenship_rule)
    else:
        add_criterion(
            "citizenship", citizenship,
            "met" if str(citizenship).casefold() in str(citizenship_rule).casefold() else "unknown",
            citizenship_rule,
        )

    demographics = student_profile.get("demographics")
    demographic_rules = row.get("demographic_focus") or []
    if not demographics or not demographic_rules:
        add_criterion("demographics", demographics, "unknown", demographic_rules)
    else:
        profile_values = demographics if isinstance(demographics, list) else [demographics]
        matches = any(
            str(value).casefold() in str(rule).casefold()
            for value in profile_values
            for rule in demographic_rules
        )
        add_criterion("demographics", demographics, "met" if matches else "unknown", demographic_rules)

    need = student_profile.get("need_based")
    need_rule = row.get("need_based")
    if need is None or need_rule is None:
        add_criterion("financial_need", need, "unknown", need_rule)
    else:
        add_criterion("financial_need", need, "met" if need is need_rule else "unknown", need_rule)

    return compact({
        "program_name": row.get("program_name"),
        "criteria": criteria,
        "source_url": row.get("source_url"),
        "crawl_date": row.get("crawl_date"),
        "evidence": evidence,
        "note": "This is a conservative corpus comparison, not an eligibility determination. Student profile is not persisted.",
    })

In [12]:
_LAST_HTTP_REQUEST = 0.0
_HTTP_INTERVAL_SECONDS = 2.0
_HTTP_SESSION = requests.Session()
_HTTP_SESSION.headers.update({"User-Agent": "ScholarshipCollegeRAG/1.0 (responsible source verification)"})

def safe_public_url(url):
    parsed = urlparse(url)
    if parsed.scheme not in {"http", "https"} or not parsed.hostname or parsed.username or parsed.password:
        return False
    host = parsed.hostname.casefold()
    if host in {"localhost", "localhost.localdomain"} or host.endswith(".local"):
        return False
    try:
        if ipaddress.ip_address(host).is_private or ipaddress.ip_address(host).is_loopback:
            return False
    except ValueError:
        pass
    return True

def wait_for_rate_limit():
    global _LAST_HTTP_REQUEST
    delay = _HTTP_INTERVAL_SECONDS - (time.monotonic() - _LAST_HTTP_REQUEST)
    if delay > 0:
        time.sleep(delay)
    _LAST_HTTP_REQUEST = time.monotonic()

def organization_name_matches_domain(record, host):
    official_domain = str(record.get("website_domain") or "").casefold()
    if official_domain:
        return host == official_domain or host.endswith("." + official_domain)
    organization = str(record.get("organization") or "")
    organization_tokens = {
        token.casefold()
        for token in re.findall(r"[A-Za-z0-9]+", organization)
        if len(token) > 2
    }
    host_labels = {label for label in host.split(".") if label and label != "www"}
    if not organization_tokens:
        return None
    return any(
        token in label or label in token
        for token in organization_tokens
        for label in host_labels
    )

@tool
def verify_official_page(url_or_program_name: str) -> str:
    "Check a corpus source URL, robots policy, redirect domain, freshness, and stored award/deadline mentions."
    matches = []
    parsed_input = urlparse(url_or_program_name)
    if parsed_input.scheme in {"http", "https"}:
        matches = [
            row for row in scholarship_records
            if row.get("source_url") == url_or_program_name
        ] + [
            row for row in college_records
            if url_or_program_name in {
                row.get("website_url"), row.get("admissions_url"),
                row.get("financial_aid_url"), row.get("scholarships_url"),
            }
        ]
    else:
        matches = matched_scholarship_rows(url_or_program_name, limit=3)
        matches += matched_college_rows(url_or_program_name, limit=3)
    if not matches:
        return compact({"checked": False, "reason": "URL or record is not in the local corpus; refusing an ungrounded fetch."})
    record = matches[0]
    url = url_or_program_name if parsed_input.scheme in {"http", "https"} else (
        record.get("source_url") or record.get("website_url")
    )
    if not url or not safe_public_url(url):
        return compact({"checked": False, "reason": "No safe public HTTP(S) source URL is available."})
    registered_url = record.get("source_url") or record.get("website_url") or url
    registered_host = (urlparse(registered_url).hostname or "").casefold()
    requested_host = (urlparse(url).hostname or "").casefold()
    if not (requested_host == registered_host or requested_host.endswith("." + registered_host)):
        return compact({"checked": False, "reason": "Requested host does not match the corpus source host."})
    owner_match = organization_name_matches_domain(record, requested_host)

    robots_url = f"{urlparse(url).scheme}://{urlparse(url).netloc}/robots.txt"
    robot_parser = RobotFileParser(robots_url)
    try:
        wait_for_rate_limit()
        robot_response = _HTTP_SESSION.get(robots_url, timeout=15, allow_redirects=True)
        robots_host = (urlparse(robot_response.url).hostname or "").casefold()
        if not (robots_host == registered_host or robots_host.endswith("." + registered_host)):
            return compact({"checked": False, "reason": "robots.txt redirected outside the registered source domain."})
        if robot_response.status_code == 200:
            robot_parser.parse(robot_response.text.splitlines())
        elif robot_response.status_code == 404:
            robot_parser.parse([])
        else:
            return compact({"checked": False, "reason": "robots.txt disallows verification."})
        if not robot_parser.can_fetch(_HTTP_SESSION.headers["User-Agent"], url):
            return compact({"checked": False, "reason": "robots.txt disallows verification."})
        wait_for_rate_limit()
        response = _HTTP_SESSION.get(url, timeout=20, allow_redirects=True)
    except requests.RequestException as exc:
        return compact({"checked": False, "error": str(exc), "source_url": url})

    final_host = (urlparse(response.url).hostname or "").casefold()
    if not (final_host == registered_host or final_host.endswith("." + registered_host)):
        return compact({
            "checked": True,
            "status_code": response.status_code,
            "final_url": response.url,
            "same_source_domain": False,
            "organization_name_matches_domain_heuristically": owner_match,
            "source_url": url,
        })
    page_text = " ".join(BeautifulSoup(response.text, "html.parser").get_text(" ").split()).casefold()
    mentions = {}
    for field in ("deadline_text", "award_amount_text"):
        expected = str(record.get(field) or "").strip()
        mentions[field] = None if not expected else expected.casefold() in page_text
    crawl_date = record.get("crawl_date") or record.get("extracted_date")
    age_days = None
    if crawl_date:
        try:
            age_days = (date.today() - date.fromisoformat(str(crawl_date)[:10])).days
        except ValueError:
            age_days = None
    result = {
        "checked": True,
        "status_code": response.status_code,
        "final_url": response.url,
        "same_source_domain": True,
        "organization_name_matches_domain_heuristically": owner_match,
        "domain_ownership_note": "The corpus has no independent sponsor-domain registry; this name/domain heuristic is not proof of ownership.",
        "stored_deadline_or_award_mentions": mentions,
        "crawl_date": crawl_date,
        "crawl_age_days": age_days,
        "stale_over_180_days": None if age_days is None else age_days > 180,
        "source_url": url,
    }
    if USE_TAVILY:
        if not TAVILY_API_KEY:
            raise RuntimeError("USE_TAVILY=True requires TAVILY_API_KEY.")
        wait_for_rate_limit()
        tavily_response = requests.post(
            "https://api.tavily.com/extract",
            json={"api_key": TAVILY_API_KEY, "urls": [url]},
            timeout=30,
        )
        tavily_response.raise_for_status()
        result["tavily_extract"] = tavily_response.json()
    return compact(result)

@tool
def find_upcoming_deadlines(within_days: int = 60, state: str | None = None, stem_field: str | None = None, applicant_stage: str | None = None, demographic_focus: str | None = None, need_based: bool | None = None) -> str:
    "List dated scholarship deadlines in a time window; flag past/missing dates and project annual recurrences."
    if within_days < 0:
        raise ValueError("within_days must be nonnegative")
    today = date.today()
    end_date = today + timedelta(days=within_days)
    filters = {
        "state": state,
        "stem_fields": stem_field,
        "applicant_stage": applicant_stage,
        "demographic_focus": demographic_focus,
        "need_based": need_based,
    }
    output = []
    past_or_missing = []
    past_or_missing_count = 0
    for row in scholarship_records:
        if row.get("page_type") == "not_a_program" or not row.get("program_name"):
            continue
        if not metadata_matches(row, filters, "scholarships"):
            continue
        raw_deadline = row.get("deadline_date_iso")
        if not raw_deadline:
            status, effective = "missing_parsed_date", None
        else:
            try:
                effective = date.fromisoformat(str(raw_deadline)[:10])
            except ValueError:
                status, effective = "invalid_parsed_date", None
            if effective is not None and effective < today:
                if row.get("deadline_recurs_annually") is True:
                    try:
                        effective = date(today.year, effective.month, effective.day)
                    except ValueError:
                        effective = date(today.year, 2, 28)
                    if effective < today:
                        effective = date(effective.year + 1, effective.month, effective.day)
                    status = "projected_recurring_verify_with_sponsor"
                else:
                    status = "past"
            elif effective is not None:
                status = "upcoming"
        if status in {"past", "missing_parsed_date", "invalid_parsed_date"}:
            past_or_missing_count += 1
            if len(past_or_missing) < 10:
                past_or_missing.append({
                    "program_name": row.get("program_name"),
                    "deadline_date_iso": raw_deadline,
                    "deadline_text": row.get("deadline_text"),
                    "deadline_status": status,
                    "source_url": row.get("source_url"),
                })
        if effective is not None and today <= effective <= end_date:
            output.append({
                "program_name": row.get("program_name"),
                "deadline_date": effective.isoformat(),
                "deadline_status": status,
                "deadline_text": row.get("deadline_text"),
                "deadline_recurs_annually": row.get("deadline_recurs_annually"),
                "crawl_date": row.get("crawl_date"),
                "source_url": row.get("source_url"),
                "evidence": row.get("evidence"),
            })
    output.sort(key=lambda item: item["deadline_date"])
    return compact({
        "today": today.isoformat(),
        "within_days": within_days,
        "results": output,
        "past_or_missing": past_or_missing,
        "past_or_missing_count": past_or_missing_count,
        "past_or_missing_truncated": past_or_missing_count > len(past_or_missing),
        "note": "Dates in deadline_text without deadline_date_iso are not silently parsed. Projected recurring dates must be verified with the sponsor.",
    })

TOOL_LIST = [
    find_colleges,
    verify_official_page,
    search_scholarships,
    evaluate_amount,
    evaluate_eligibility,
    find_upcoming_deadlines,
]
print("Registered tools:", [item.name for item in TOOL_LIST])

Registered tools: ['find_colleges', 'verify_official_page', 'search_scholarships', 'evaluate_amount', 'evaluate_eligibility', 'find_upcoming_deadlines']


# 8. System prompt

The required base sentence is kept verbatim. The separated rules add citations, freshness, abstention, privacy, and advice limitations.

In [13]:
REQUIRED_BASE_PROMPT = "You are a helpful AI student college and scholarship research assistant. Use the tools to gather facts and answer the user queries. Ask focused follow-ups if data is missing. Be concise and friendly."
SYSTEM_PROMPT = REQUIRED_BASE_PROMPT
SYSTEM_RULES = (
    "Rules:\n"
    "- Ground factual claims in retrieved corpus evidence. Cite source_url and include crawl_date for deadlines.\n"
    "- A null or missing field means unknown; say when information is not in the knowledge base.\n"
    "- Never invent awards, eligibility, renewal terms, college programs, application links, or deadlines.\n"
    "- Mark stale deadlines and recurring-date projections; tell the student to verify on the sponsor's official page.\n"
    "- College directory rows and pending STEM/funding verification are not proof of programs or awards.\n"
    "- Ask one focused follow-up when required profile facts are missing; abstain when evidence is absent.\n"
    "- Do not persist or transmit student-profile details; the profile exists only for the current tool call.\n"
    "- This is not financial or legal advice. Do not submit applications.\n"
)

def system_prompt_for_today():
    return SYSTEM_PROMPT + "\n\n" + SYSTEM_RULES.strip() + f"\n\nToday's date: {date.today().isoformat()}."

assert SYSTEM_PROMPT.startswith(REQUIRED_BASE_PROMPT)

# 9. Tool loadout

`TOP_K=5` is the number of retrieved chunks; `TOOL_TOP_K=3` is the number of tools passed to the agent. With only six tools, a tool top-k of five removes little context, so the default is three. If every tool-description similarity is below the threshold, the safe fallback deliberately returns all six tools; that exceptional fallback can exceed `TOOL_TOP_K`.

In [14]:
TOOL_TOP_K = 3
TOOL_SIMILARITY_THRESHOLD = 0.05
_TOOL_VECTORS = None

def cosine_similarity(left, right):
    numerator = sum(a * b for a, b in zip(left, right, strict=True))
    left_norm = math.sqrt(sum(value * value for value in left))
    right_norm = math.sqrt(sum(value * value for value in right))
    return numerator / (left_norm * right_norm) if left_norm and right_norm else 0.0

def select_scored_tools(scored):
    fallback = scored[0][0] < TOOL_SIMILARITY_THRESHOLD
    selected = TOOL_LIST if fallback else [tool_item for _, tool_item in scored[:TOOL_TOP_K]]
    return selected, fallback

def select_tools(query):
    global _TOOL_VECTORS
    descriptions = [f"{item.name}: {item.description}" for item in TOOL_LIST]
    vectors = get_embeddings()
    if _TOOL_VECTORS is None:
        _TOOL_VECTORS = vectors.embed_documents(descriptions)
    query_vector = vectors.embed_query(query)
    scored = sorted(
        [
            (cosine_similarity(query_vector, vector), tool_item)
            for tool_item, vector in zip(TOOL_LIST, _TOOL_VECTORS, strict=True)
        ],
        key=lambda item: item[0],
        reverse=True,
    )
    selected, fallback = select_scored_tools(scored)
    logger.info(
        "tool loadout fallback=%s selected=%s scores=%s",
        fallback,
        [item.name for item in selected],
        [(item.name, round(score, 4)) for score, item in scored],
    )
    return selected

# 10. Building the agent

This uses the reference app’s `create_agent` construction style with a Nebius-backed `ChatOpenAI` client. A recursion limit bounds the tool-call loop. Errors are logged and returned as explicit, non-success-shaped results.

In [15]:
def build_agent(selected_tools):
    "Build a LangChain tool-calling agent with the configured Nebius chat model."
    model = ChatOpenAI(
        model=required_env("NEBIUS_LLM_MODEL"),
        api_key=required_env("NEBIUS_API_KEY"),
        base_url=required_env("NEBIUS_BASE_URL").rstrip("/") + "/",
        temperature=0,
    )
    return create_agent(
        model=model,
        tools=selected_tools,
        system_prompt=system_prompt_for_today(),
    )

def tool_calls_from_state(state):
    calls = []
    for message in state.get("messages", []):
        for call in getattr(message, "tool_calls", None) or []:
            calls.append({"name": call.get("name"), "args": call.get("args", {})})
    return calls

def token_usage_from_state(state):
    totals = {"input_tokens": 0, "output_tokens": 0}
    for message in state.get("messages", []):
        usage = getattr(message, "usage_metadata", None) or {}
        totals["input_tokens"] += usage.get("input_tokens", 0)
        totals["output_tokens"] += usage.get("output_tokens", 0)
    return totals

def handle_request(query, use_loadout=True):
    if DRY_RUN:
        raise RuntimeError("Set DRY_RUN=False after reviewing costs/configuration before invoking an agent.")
    started = time.perf_counter()
    try:
        selected = select_tools(query) if use_loadout else TOOL_LIST
        agent = build_agent(selected)
        state = agent.invoke(
            {"messages": [{"role": "user", "content": query}]},
            config={"recursion_limit": 16},
        )
        messages = state.get("messages", [])
        answer = messages[-1].content if messages else ""
        return {
            "answer": answer,
            "selected_tools": [item.name for item in selected],
            "selection_mode": "loadout" if use_loadout else "all_tools",
            "tool_calls": tool_calls_from_state(state),
            "token_usage": token_usage_from_state(state),
            "latency_seconds": time.perf_counter() - started,
            "error": None,
        }
    except Exception as exc:
        logger.exception("Agent request failed")
        return {
            "answer": "I could not complete this request because the agent or a provider failed. Please retry after checking the logged error.",
            "selected_tools": [],
            "selection_mode": "loadout" if use_loadout else "all_tools",
            "tool_calls": [],
            "token_usage": {"input_tokens": 0, "output_tokens": 0},
            "latency_seconds": time.perf_counter() - started,
            "error": f"{type(exc).__name__}: {exc}",
        }

# 11. DEMO Queries and evaluation harness

The list has 18 rows: 4 straightforward; 3 multi-document; 3 filtered discovery; 2 ambiguous; 3 intentionally unanswerable; and 3 college-research questions. College examples are selected from the actual directory corpus. The College Scholarship Program has a text deadline but no parsed date; the no-essay and state/major filters may reveal schema gaps rather than a match.

Evaluation is opt-in (`RUN_DEMOS=False`). When enabled after ingestion, it records selected/actual tools, retrieved chunk IDs and scores, answer, latency, token usage, retrieval-hit flag, abstention heuristic, and a blank manual-review field. Quality is not automatically judged.

In [16]:
DEMO_QUERIES = [
    {"id": "Q01", "category": "straightforward", "query": "Give me an overview of the Edison Scholars award and who it is for.", "expected_behavior": "Summarize only stored award and eligibility facts; cite the source and note null deadline data.", "expected_tools": ["search_scholarships"], "expected_sources": ["Edison Scholars"]},
    {"id": "Q02", "category": "straightforward", "query": "What amount does the SMART Scholarship-for-Service Program list, and does it vary?", "expected_behavior": "Return stored amount range/text and explain missing renewal values as unknown.", "expected_tools": ["evaluate_amount"], "expected_sources": ["SMART Scholarship-for-Service Program"]},
    {"id": "Q03", "category": "straightforward", "query": "Can the Edison Scholars award be renewed, and what renewal terms are documented?", "expected_behavior": "Report only the explicit renewable value and source; distinguish a multi-year amount statement from renewal rules.", "expected_tools": ["evaluate_amount"], "expected_sources": ["Edison Scholars"]},
    {"id": "Q04", "category": "straightforward", "query": "What is the next parsed deadline for the National Space Club and Foundation Keynote Scholarship?", "expected_behavior": "Use the stored parsed date, show crawl date, and warn that sponsor verification is required.", "expected_tools": ["find_upcoming_deadlines", "search_scholarships"], "expected_sources": ["National Space Club and Foundation Keynote Scholarship"]},
    {"id": "Q05", "category": "multi_document", "query": "Compare the documented award amounts and eligibility evidence for Edison Scholars and SMART Scholarship-for-Service.", "expected_behavior": "Compare separate source records; do not imply awards can be stacked.", "expected_tools": ["search_scholarships", "evaluate_amount"], "expected_sources": ["Edison Scholars", "SMART Scholarship-for-Service Program"]},
    {"id": "Q06", "category": "multi_document", "query": "What does the corpus say about Edison Scholars and Carnegie Mellon University, and is the scholarship explicitly tied to that college?", "expected_behavior": "Use scholarship and college records; state that the corpus does not establish an award-college affiliation.", "expected_tools": ["search_scholarships", "find_colleges"], "expected_sources": ["Edison Scholars", "Carnegie Mellon University"]},
    {"id": "Q07", "category": "multi_document", "query": "Which scholarships in this corpus can I use at Carnegie Mellon University?", "expected_behavior": "Do not infer college-specific use; explain the college record has no verified scholarship inventory and ask the user to verify with sponsors.", "expected_tools": ["find_colleges", "search_scholarships"], "expected_sources": ["Carnegie Mellon University"]},
    {"id": "Q08", "category": "filtered_discovery", "query": "Find no-essay STEM scholarships for a Virginia high-school senior with a parsed deadline in the next 60 days.", "expected_behavior": "Apply supported filters; explain if residency or STEM fields are unrecorded and avoid claiming eligibility.", "expected_tools": ["search_scholarships", "find_upcoming_deadlines"], "expected_sources": ["Virginia Academic Scholarship", "Scholarships360"]},
    {"id": "Q09", "category": "filtered_discovery", "query": "As a California senior interested in engineering, show no-essay awards due in the next 60 days.", "expected_behavior": "Return only criteria evidenced by corpus; note that state restrictions and parsed dates may be missing.", "expected_tools": ["search_scholarships", "find_upcoming_deadlines"], "expected_sources": ["Edison Scholars", "Scholarships360"]},
    {"id": "Q10", "category": "filtered_discovery", "query": "Find no-essay math scholarships for a high-school junior in Maryland with deadlines in the next 30 days.", "expected_behavior": "Use filters conservatively and report no supported match rather than relaxing missing criteria silently.", "expected_tools": ["search_scholarships", "find_upcoming_deadlines"], "expected_sources": []},
    {"id": "Q11", "category": "ambiguous", "query": "Can I qualify for any scholarship?", "expected_behavior": "Ask one focused follow-up for grade, state, citizenship, GPA, major, or target program before personal screening.", "expected_tools": ["search_scholarships", "evaluate_eligibility"], "expected_sources": []},
    {"id": "Q12", "category": "ambiguous", "query": "Is this enough money for me?", "expected_behavior": "Ask which named scholarship and what attendance-cost context the user means; do not infer affordability.", "expected_tools": ["evaluate_amount"], "expected_sources": []},
    {"id": "Q13", "category": "unanswerable", "query": "What is the deadline and award amount for the Gates Cambridge Scholarship in this corpus?", "expected_behavior": "Abstain: no matching program is present in the inspected scholarship corpus.", "expected_tools": ["search_scholarships", "evaluate_amount"], "expected_sources": ["Gates Cambridge Scholarship"]},
    {"id": "Q14", "category": "unanswerable", "query": "What is the application deadline for the Nicholas H. Colton Memorial Scholarship?", "expected_behavior": "Say the parsed deadline and deadline text are missing; do not infer a date.", "expected_tools": ["search_scholarships", "find_upcoming_deadlines"], "expected_sources": ["Nicholas H. Colton Memorial Scholarship"]},
    {"id": "Q15", "category": "unanswerable", "query": "What engineering scholarships does the University of Atlantis offer?", "expected_behavior": "Abstain: the college is absent and no award affiliation can be established.", "expected_tools": ["find_colleges", "search_scholarships"], "expected_sources": ["University of Atlantis"]},
    {"id": "Q16", "category": "college_research", "query": "What official website, admissions page, and financial-aid page are listed for the Massachusetts Institute of Technology?", "expected_behavior": "Return directory URLs and location from the college record; do not infer program or award details.", "expected_tools": ["find_colleges"], "expected_sources": ["Massachusetts Institute of Technology"]},
    {"id": "Q17", "category": "college_research", "query": "Find colleges in California and show the official website and admissions link for Stanford University.", "expected_behavior": "Use the state filter and cite the directory record; note that the corpus does not verify STEM offerings.", "expected_tools": ["find_colleges"], "expected_sources": ["Stanford University"]},
    {"id": "Q18", "category": "college_research", "query": "Compare the directory information for Carnegie Mellon University and MIT: state, official website, admissions, financial aid, and verified scholarship availability.", "expected_behavior": "Compare recorded directory fields only; state that neither record verifies a scholarship listing and STEM programs are pending verification.", "expected_tools": ["find_colleges"], "expected_sources": ["Carnegie Mellon University", "Massachusetts Institute of Technology"]},
]
assert len(DEMO_QUERIES) == 18
assert pd.Series([q["category"] for q in DEMO_QUERIES]).value_counts().to_dict() == {
    "straightforward": 4, "multi_document": 3, "filtered_discovery": 3,
    "ambiguous": 2, "unanswerable": 3, "college_research": 3,
}

In [17]:
def answer_looks_like_abstention(answer):
    text = str(answer).casefold()
    cues = (
        "not in the knowledge base", "not found in the corpus",
        "does not establish", "cannot verify", "i don't have",
        "not available in", "unknown",
    )
    return any(cue in text for cue in cues)

def expected_source_hit(results, expected_sources):
    if not expected_sources:
        return None
    for result in results:
        metadata = result["metadata"]
        haystack = " ".join(str(metadata.get(key, "")) for key in (
            "program_name", "college_name", "source_url", "display_name"
        )).casefold()
        if any(str(source).casefold() in haystack for source in expected_sources):
            return True
    return False

def run_evaluation(queries, use_loadout=True):
    rows = []
    for item in queries:
        previews = []
        for namespace in NAMESPACES:
            for hit in retrieve(item["query"], namespace, k=TOP_K):
                previews.append({
                    "namespace": namespace,
                    "chunk_id": hit["chunk_id"],
                    "score": hit["score"],
                    "metadata": hit["metadata"],
                })
        result = handle_request(item["query"], use_loadout=use_loadout)
        called = [call["name"] for call in result["tool_calls"]]
        rows.append({
            "id": item["id"],
            "category": item["category"],
            "mode": result["selection_mode"],
            "tools_selected": result["selected_tools"],
            "tools_called": called,
            "top_k_chunks": [
                {"namespace": chunk["namespace"], "chunk_id": chunk["chunk_id"], "score": chunk["score"]}
                for chunk in previews
            ],
            "answer": result["answer"],
            "latency_seconds": result["latency_seconds"],
            "input_tokens": result["token_usage"]["input_tokens"],
            "output_tokens": result["token_usage"]["output_tokens"],
            "retrieval_hit": expected_source_hit(previews, item["expected_sources"]),
            "abstention_flag": answer_looks_like_abstention(result["answer"]),
            "manual_review": "",
            "expected_tools": item["expected_tools"],
            "tool_call_trace": result["tool_calls"],
            "error": result["error"],
        })
    frame = pd.DataFrame(rows)
    EVALUATION_PATH.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(EVALUATION_PATH, index=False)
    return frame

if RUN_DEMOS:
    if DRY_RUN:
        raise RuntimeError("Set DRY_RUN=False before running demos.")
    evaluation_df = run_evaluation(DEMO_QUERIES, use_loadout=True)
    display(evaluation_df)
    print(f"Saved evaluation to {EVALUATION_PATH}")
else:
    print("Demo/evaluation calls skipped. Set DRY_RUN=False and RUN_DEMOS=True to enable.")

INFO HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
INFO HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
INFO HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
INFO HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
INFO tool loadout fallback=False selected=['evaluate_amount', 'search_scholarships', 'evaluate_eligibility'] scores=[('evaluate_amount', 0.6872), ('search_scholarships', 0.6782), ('evaluate_eligibility', 0.588), ('find_upcoming_deadlines', 0.5775), ('find_colleges', 0.5654), ('verify_official_page', 0.4338)]
INFO HTTP Request: POST https://api.tokenfactory.nebius.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
INFO HTTP Request: POST https://api.tokenfactory.nebius.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO HTTP Request: POST https://ap

,id,category,mode,tools_selected,tools_called,top_k_chunks,answer,latency_seconds,input_tokens,output_tokens,retrieval_hit,abstention_flag,manual_review,expected_tools,tool_call_trace,error
0,Q01,straightforward,loadout,"[evaluate_amount, search_scholarships, evaluat...",[search_scholarships],"[{'namespace': 'scholarships', 'chunk_id': '17...",**Edison Scholars – Overview**\n\n| Item | Det...,9.665322,2123,798,True,False,,[search_scholarships],"[{'name': 'search_scholarships', 'args': {'que...",NaN
1,Q02,straightforward,loadout,"[evaluate_amount, search_scholarships, find_up...",[evaluate_amount],"[{'namespace': 'scholarships', 'chunk_id': 'cb...",The SMART Scholarship‑for‑Service Program list...,1.590246,3395,244,True,False,,[evaluate_amount],"[{'name': 'evaluate_amount', 'args': {'program...",NaN
2,Q03,straightforward,loadout,"[evaluate_amount, search_scholarships, find_up...",[evaluate_amount],"[{'namespace': 'scholarships', 'chunk_id': '17...",**Edison Scholars (Edison International)** \n...,3.853102,3158,513,True,False,,[evaluate_amount],"[{'name': 'evaluate_amount', 'args': {'program...",NaN
3,Q04,straightforward,loadout,"[find_upcoming_deadlines, evaluate_amount, sea...",[search_scholarships],"[{'namespace': 'scholarships', 'chunk_id': '69...",**National Space Club and Foundation Keynote S...,1.858127,2852,296,True,False,,"[find_upcoming_deadlines, search_scholarships]","[{'name': 'search_scholarships', 'args': {'que...",NaN
4,Q05,multi_document,loadout,"[evaluate_amount, search_scholarships, evaluat...","[search_scholarships, search_scholarships]","[{'namespace': 'scholarships', 'chunk_id': '17...",**Edison Scholars (Edison International)** \n...,4.547128,5592,1258,True,False,,"[search_scholarships, evaluate_amount]","[{'name': 'search_scholarships', 'args': {'que...",NaN
5,Q06,multi_document,loadout,[],[],"[{'namespace': 'scholarships', 'chunk_id': '17...",I could not complete this request because the ...,7.493683,0,0,True,False,,"[search_scholarships, find_colleges]",[],GraphRecursionError: Recursion limit of 16 rea...
6,Q07,multi_document,loadout,"[search_scholarships, evaluate_amount, find_co...",[search_scholarships],"[{'namespace': 'scholarships', 'chunk_id': '7d...",Below are the scholarships in the current corp...,5.809773,5384,2015,True,False,,"[find_colleges, search_scholarships]","[{'name': 'search_scholarships', 'args': {'que...",NaN
7,Q08,filtered_discovery,loadout,"[search_scholarships, find_upcoming_deadlines,...","[find_upcoming_deadlines, search_scholarships]","[{'namespace': 'scholarships', 'chunk_id': '96...","I wasn’t able to locate any **STEM‑focused, no...",4.290606,3310,904,True,False,,"[search_scholarships, find_upcoming_deadlines]","[{'name': 'find_upcoming_deadlines', 'args': {...",NaN
8,Q09,filtered_discovery,loadout,"[evaluate_amount, find_upcoming_deadlines, sea...",[find_upcoming_deadlines],"[{'namespace': 'scholarships', 'chunk_id': '48...",**No‑Essay Scholarship Due in the Next 60 Days...,3.665297,4532,764,True,False,,"[search_scholarships, find_upcoming_deadlines]","[{'name': 'find_upcoming_deadlines', 'args': {...",NaN
9,Q10,filtered_discovery,loadout,"[search_scholarships, evaluate_amount, find_up...","[search_scholarships, find_upcoming_deadlines]","[{'namespace': 'scholarships', 'chunk_id': '7d...",I wasn’t able to locate any **no‑essay math sc...,3.146304,2057,593,None,False,,"[search_scholarships, find_upcoming_deadlines]","[{'name': 'search_scholarships', 'args': {'que...",NaN


Saved evaluation to c:\Users\Rosh\GENAI\WEEK2_PROJECT\college_resources_for_high_schoolers_STEM\reports\rag_evaluation.csv


# 12. Running Without Tool Loadout (All Tools)

The comparison runs the same 18 queries with all six tools attached. It reports tool-selection precision/recall against the pre-registered `expected_tools`, input tokens, latency, retrieval-hit/abstention rates, and a manual answer-quality field. The manual column must be reviewed by a person; it is not an LLM-graded truth score.

In [91]:
def tool_precision_recall(row):
    expected = set(row["expected_tools"])
    selected = set(row["tools_selected"])
    precision = len(expected & selected) / len(selected) if selected else 0.0
    recall = len(expected & selected) / len(expected) if expected else 1.0
    return precision, recall

if RUN_DEMOS:
    all_tools_df = run_evaluation(DEMO_QUERIES, use_loadout=False)
    combined_runs = pd.concat([evaluation_df, all_tools_df], ignore_index=True)
    metrics = []
    for mode, group in combined_runs.groupby("mode"):
        scores = group.apply(tool_precision_recall, axis=1)
        metrics.append({
            "mode": mode,
            "queries": len(group),
            "mean_tool_precision": sum(score[0] for score in scores) / len(scores),
            "mean_tool_recall": sum(score[1] for score in scores) / len(scores),
            "mean_input_tokens": group["input_tokens"].mean(),
            "mean_latency_seconds": group["latency_seconds"].mean(),
            "retrieval_hit_rate": group["retrieval_hit"].dropna().mean(),
            "abstention_rate": group["abstention_flag"].mean(),
            "answer_quality": "manual review required",
        })
    comparison_df = pd.DataFrame(metrics)
    display(comparison_df)
else:
    print("Comparison skipped; it uses the same opt-in RUN_DEMOS flag and requires provider calls.")

INFO HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
INFO HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
INFO HTTP Request: POST https://api.tokenfactory.nebius.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
INFO HTTP Request: POST https://api.tokenfactory.nebius.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
INFO HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
INFO HTTP Request: POST https://api.tokenfactory.nebius.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"
INFO HTTP Request: POST https://api.tokenfactory.nebius.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO HTTP Request: POST https://api.tokenfactory.nebius.

,mode,queries,mean_tool_precision,mean_tool_recall,mean_input_tokens,mean_latency_seconds,retrieval_hit_rate,abstention_rate,answer_quality
0,all_tools,18,0.212963,0.833333,8788.666667,5.854258,0.8,0.0,manual review required
1,loadout,18,0.500000,0.944444,10092.500000,8.779019,0.8,0.0,manual review required


# 13. Findings and failure analysis

**Pre-registered failure predictions (compare after the run):**
- Acronyms/abbreviations (for example SWE, NSBE, STEM subfields) may rank poorly unless the source text spells them out.
- Missing `deadline_date_iso`, null renewal fields, and stale crawl dates will cause abstentions or sponsor-verification warnings.
- Recurring deadlines may be projected from historical dates; those projections are never presented as confirmed.
- College rows are discovery records only: `scholarships_url` is null for every record and STEM program verification is pending, so college-specific award/program claims should fail closed.
- State, essay, demographic, and major filters can be too strict because scholarship records do not have a dedicated state field and many criteria are null or empty.
- Duplicate names/source pages can create apparently conflicting award rows; report each source and avoid automatic award stacking or silent merging.
- Chunk boundaries, weak source metadata, and embedding-model mismatch may lower retrieval-hit rates or lose evidence context.

Fill this table after running the evaluation; the notebook intentionally does not fabricate results.

| Query ID | Retrieval success / partial / failure | Why (null deadline, stale crawl, filter, split context, acronym, etc.) | Fix |
|---|---|---|---|
| Q01 | Not run | | |
| Q02 | Not run | | |
| Q03 | Not run | | |
| Q04 | Not run | | |
| Q05 | Not run | | |
| Q06 | Not run | | |
| Q07 | Not run | | |
| Q08 | Not run | | |
| Q09 | Not run | | |
| Q10 | Not run | | |
| Q11 | Not run | | |
| Q12 | Not run | | |
| Q13 | Not run | | |
| Q14 | Not run | | |
| Q15 | Not run | | |

# 14. Final validation

The following cell is a code-only validation checklist for a later user run. It validates corpus/chunk shape locally. Pinecone counts are checked only after `DRY_RUN=False` and successful ingestion; no agent, model, or API call is made by this cell.

In [98]:
def pinecone_safe(value):
    if value is None:
        return False
    if isinstance(value, (str, bool, int, float)):
        return True
    if isinstance(value, list):
        return all(isinstance(item, str) for item in value)
    return False

def pass_fail(task, passed, detail):
    return {"task": task, "status": "PASS" if passed else "FAIL", "detail": detail}

validation = []
both_chunked = all(chunks_by_namespace.get(name) for name in NAMESPACES)
all_chunk_ids = all(doc.metadata.get("chunk_id") for group in chunks_by_namespace.values() for doc in group)
safe_metadata = all(
    all(pinecone_safe(value) for value in doc.metadata.values())
    for group in chunks_by_namespace.values()
    for doc in group
)
validation.append(pass_fail(
    "Task 1",
    both_chunked and all_chunk_ids and safe_metadata,
    f"scholarship_chunks={len(chunks_by_namespace['scholarships'])}; college_chunks={len(chunks_by_namespace['colleges'])}; IDs and metadata types checked",
))

tools_typed = all(
    item.description
    and item.func.__doc__
    and all(parameter.annotation is not parameter.empty for parameter in __import__("inspect").signature(item.func).parameters.values())
    for item in TOOL_LIST
)
validation.append(pass_fail("Task 2", len(TOOL_LIST) == 6 and tools_typed, f"registered={[item.name for item in TOOL_LIST]}"))

if DRY_RUN:
    validation.append(pass_fail("Task 3", True, "SKIPPED AS EXPECTED: DRY_RUN=True; no Pinecone calls made"))
else:
    stats = connect_pinecone().describe_index_stats()
    namespace_counts = {name: int(stats.namespaces.get(name).vector_count) if stats.namespaces.get(name) else 0 for name in NAMESPACES}
    counts_match = all(namespace_counts[name] == len(chunks_by_namespace[name]) for name in NAMESPACES)
    validation.append(pass_fail("Task 3", counts_match, f"namespace counts={namespace_counts}"))

validation.append(pass_fail("Task 4", SYSTEM_PROMPT.startswith(REQUIRED_BASE_PROMPT), "system prompt begins with required exact base sentence"))
agent_built = False
if all(os.getenv(name, "").strip() for name in ("NEBIUS_API_KEY", "NEBIUS_BASE_URL", "NEBIUS_LLM_MODEL")):
    _validation_agent = build_agent(TOOL_LIST[:1])
    agent_built = _validation_agent is not None
validation.append(pass_fail("Task 5", callable(build_agent) and callable(handle_request) and agent_built, "Nebius create_agent instantiated without invoking the model" if agent_built else "Configure Nebius credentials; agent construction was not attempted"))
retrieval_ok = len(retrieve("Edison Scholars", "scholarships", k=TOP_K)) <= TOP_K
normal_selection, fallback_selection = select_scored_tools(
    [(1.0 - index * 0.01, item) for index, item in enumerate(TOOL_LIST)]
)
fallback_tools, fallback_used = select_scored_tools(
    [(0.0, item) for item in TOOL_LIST]
)
selection_ok = len(normal_selection) <= TOOL_TOP_K and fallback_used and len(fallback_tools) == len(TOOL_LIST)
validation.append(pass_fail("Task 6", TOP_K == 5 and TOOL_TOP_K == 3 and retrieval_ok and selection_ok, f"TOP_K={TOP_K}; TOOL_TOP_K={TOOL_TOP_K}; retrieval_count<=TOP_K; fallback returns all tools by design"))

categories = pd.Series([item["category"] for item in DEMO_QUERIES]).value_counts().to_dict()
expected_categories = {
    "straightforward": 4, "multi_document": 3, "filtered_discovery": 3,
    "ambiguous": 2, "unanswerable": 3, "college_research": 3,
}
validation.append(pass_fail("Task 7", len(DEMO_QUERIES) == 18 and categories == expected_categories, f"query categories={categories}"))
if DRY_RUN:
    validation.append(pass_fail("Task 3", True, "SKIPPED AS EXPECTED: DRY_RUN=True; no Pinecone calls made"))
else:
    stats = connect_pinecone().describe_index_stats()
    namespace_counts = {
        name: int(stats.namespaces.get(name).vector_count)
        if stats.namespaces.get(name) else 0
        for name in NAMESPACES
    }
    expected_counts = {
        name: len({doc.metadata["chunk_id"] for doc in chunks_by_namespace[name]})
        for name in NAMESPACES
    }
    duplicate_counts = {
        name: len(chunks_by_namespace[name]) - expected_counts[name]
        for name in NAMESPACES
    }
    counts_match = all(
        namespace_counts[name] == expected_counts[name]
        for name in NAMESPACES
    )
    validation.append(pass_fail(
        "Task 3",
        counts_match,
        f"namespace counts={namespace_counts}; expected unique IDs={expected_counts}; "
        f"duplicate chunk IDs={duplicate_counts}",
    ))
validation_df = pd.DataFrame(validation)
display(validation_df)
validation_df = validation_df.drop_duplicates(subset=["task"], keep="last")
display(validation_df)
assert (validation_df["status"] == "PASS").all(), validation_df[validation_df["status"] == "FAIL"]


INFO HTTP Request: POST https://api.tokenfactory.nebius.com/v1/embeddings "HTTP/1.1 200 OK"


,task,status,detail
0,Task 1,PASS,scholarship_chunks=76; college_chunks=2191; ID...
1,Task 2,PASS,"registered=['find_colleges', 'verify_official_..."
2,Task 3,FAIL,"namespace counts={'scholarships': 75, 'college..."
3,Task 4,PASS,system prompt begins with required exact base ...
4,Task 5,PASS,Nebius create_agent instantiated without invok...
5,Task 6,PASS,TOP_K=5; TOOL_TOP_K=3; retrieval_count<=TOP_K;...
6,Task 7,PASS,"query categories={'straightforward': 4, 'multi..."
7,Task 3,PASS,"namespace counts={'scholarships': 75, 'college..."


,task,status,detail
0,Task 1,PASS,scholarship_chunks=76; college_chunks=2191; ID...
1,Task 2,PASS,"registered=['find_colleges', 'verify_official_..."
3,Task 4,PASS,system prompt begins with required exact base ...
4,Task 5,PASS,Nebius create_agent instantiated without invok...
5,Task 6,PASS,TOP_K=5; TOOL_TOP_K=3; retrieval_count<=TOP_K;...
6,Task 7,PASS,"query categories={'straightforward': 4, 'multi..."
7,Task 3,PASS,"namespace counts={'scholarships': 75, 'college..."
